# Business Case: Kiva Lending Patterns - Exploratory Data Analysis and Interactive Dashboard

## Project Objective

Kiva's historical lending data is explored to identify patterns in loan distribution across sectors and countries, and to investigate differences between requested and funded loan amounts. Data inspection and cleaning, exploratory data visualization, and an interactive dashboard developed using Dash are included in this analysis.

## Note 
A brief setup section at the beginning ensures that the latest versions of all necessary libraries are installed and imported, and the paths for the data files are correctly set. Adjust this section and the file paths as needed to match your local environment.

---

## Section 1: Setup

All required libraries are installed or upgraded. Then the necessary libraries are imported.

In [385]:

%pip install --upgrade pandas numpy seaborn plotly dash


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [386]:


import os
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np


File paths for the raw input data and the cleaned output data are defined below. These paths are used in the subsequent data processing and analysis steps.

In [387]:


# Update DATA_INPUT_PATH if running from a different directory
DATA_INPUT_PATH   = os.path.join('..', 'data', 'data_abschlussprojekt.csv')
DATA_CLEANED_PATH = os.path.join('..', 'data', 'kiva_cleaned.csv')
print('Paths set.')


Paths set.


---

## Section 2: Data Loading and Cleaning

The raw dataset is loaded and inspected. Cleaning steps are applied in sequence, and the cleaned CSV is saved.

### 1. Loading the Dataset

The file uses `#` as the column separator. The first column is an unnamed row index; `index_col=0` is used so that it becomes the DataFrame index. `skipinitialspace=True` is applied so that leading spaces after each `#` are stripped from the header, yielding clean column names without further post-processing.

In [388]:
# Load the dataset

df = pd.read_csv(
    DATA_INPUT_PATH,
    sep='#',
    index_col=0,
    skipinitialspace=True
)

# print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head(5)

,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,term_in_months,lender_count,borrower_genders,repayment_interval
0,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,12.0,12,female,irregular
1,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,11.0,14,"female, female",irregular
2,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,43.0,6,female,bullet
3,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,11.0,8,female,irregular
4,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,14.0,16,female,monthly


### 2. Initial Inspection

Before any cleaning, the raw state of the data is inspected: column names, data types, first rows, summary statistics, missing values, and duplicates.

In [389]:
# Display the shape of the DataFrame
df.shape

(671205, 13)

In [390]:
# Display the column names of the DataFrame
df.columns.to_list()  # Convert the column names to a list for easier inspection

['funded_amount',
 'loan_amount',
 'activity',
 'sector',
 'use',
 'country_code',
 'country',
 'region',
 'currency',
 'term_in_months',
 'lender_count',
 'borrower_genders',
 'repayment_interval']

In [391]:
# Display the data types of each column in the DataFrame
df.dtypes

funded_amount         float64
loan_amount           float64
activity                  str
sector                    str
use                       str
country_code              str
country                   str
region                    str
currency                  str
term_in_months        float64
lender_count            int64
borrower_genders          str
repayment_interval        str
dtype: object

In [392]:
# Display concise summary of the DataFrame, including column data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 671205 entries, 0 to 671204
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   funded_amount       671205 non-null  float64
 1   loan_amount         671205 non-null  float64
 2   activity            671205 non-null  str    
 3   sector              671205 non-null  str    
 4   use                 666972 non-null  str    
 5   country_code        671197 non-null  str    
 6   country             671205 non-null  str    
 7   region              614405 non-null  str    
 8   currency            671205 non-null  str    
 9   term_in_months      671205 non-null  float64
 10  lender_count        671205 non-null  int64  
 11  borrower_genders    666984 non-null  str    
 12  repayment_interval  671205 non-null  str    
dtypes: float64(3), int64(1), str(9)
memory usage: 66.6 MB


In [393]:
# Display summary statistics for all columns, including categorical and numerical data
df.describe(include='all')

KeyboardInterrupt: 

#### Missing Values

The count and percentage of missing values are reported per column.

In [ ]:
# Calculate and display missing values for each column in the DataFrame
# Calculate missing counts and percentages
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

# Combine into a new DataFrame
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})

# Filter to show only columns that have at least one missing value
missing_df = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)

# Display the result
print("Columns with missing values:")
display(missing_df)


Columns with missing values:


,Missing Count,Missing %
region,56800,8.46
use,4233,0.63
borrower_genders,4221,0.63
country_code,8,0.00


#### Duplicate Rows

In [ ]:
# Display the count of unique values for each column
df.nunique()

funded_amount            610
loan_amount              479
activity                 163
sector                    15
use                   423452
country_code              86
country                   87
region                 12695
currency                  67
term_in_months           148
lender_count             503
borrower_genders       11298
repayment_interval         4
dtype: int64

In [ ]:
# Display all duplicated rows in the DataFrame. Mark all duplicates (The first occurrence is also marked). 
df[df.duplicated(keep=False)]

,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,term_in_months,lender_count,borrower_genders,repayment_interval
327,275.0,275.0,Farming,Agriculture,to buy fertilizers and other farm supplies.,PH,Philippines,"Brookes Point, Palawan",PHP,8.0,8,female,irregular
392,100.0,100.0,Home Energy,Personal Use,to buy a solar lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
405,100.0,100.0,Home Energy,Personal Use,to buy a solar-powered lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
498,100.0,100.0,Home Energy,Personal Use,to buy a solar-powered lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
606,100.0,100.0,Home Energy,Personal Use,to buy a solar-powered lamp.,SV,El Salvador,NaN,USD,14.0,4,male,monthly
...,...,...,...,...,...,...,...,...,...,...,...,...,...
671200,0.0,25.0,Livestock,Agriculture,"[True, u'para compara: cemento, arenya y ladri...",PY,Paraguay,Concepción,USD,13.0,0,female,monthly
671201,25.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,13.0,1,female,monthly
671202,0.0,25.0,Games,Entertainment,NaN,KE,Kenya,NaN,KES,13.0,0,NaN,monthly
671203,0.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,13.0,0,female,monthly


In [ ]:
# Check for exact duplicate rows in the DataFrame. Because this excludes the first occurrence, the count differs from the output of the previous cell that marked all duplicates.
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes:,}")  
print(f"Unique rows: {len(df) - n_dupes:,} of {len(df):,} ({(1 - n_dupes/len(df))*100:.2f}%)")  # Print the number and percentage of unique rows

Exact duplicate rows: 24,588
Unique rows: 646,617 of 671,205 (96.34%)


#### Categorical Column Values

Unique values in key categorical columns are checked to identify encoding inconsistencies before cleaning.

In [ ]:
print("Unique sectors:", sorted(df['sector'].dropna().unique()))
print("\nUnique repayment intervals:", sorted(df['repayment_interval'].dropna().unique()))

Unique sectors: ['Agriculture', 'Arts', 'Clothing', 'Construction', 'Education', 'Entertainment', 'Food', 'Health', 'Housing', 'Manufacturing', 'Personal Use', 'Retail', 'Services', 'Transportation', 'Wholesale']

Unique repayment intervals: ['bullet', 'irregular', 'monthly', 'weekly']


#### Numerical Column Ranges

Columns that should be strictly positive are checked for negative or zero values.

In [ ]:
for col in ['funded_amount', 'loan_amount', 'term_in_months', 'lender_count']:
    neg = (df[col] < 0).sum()
    zero = (df[col] == 0).sum()
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, zeros={zero:,}, negatives={neg:,}")

funded_amount: min=0.0, max=100000.0, zeros=3,383, negatives=0
loan_amount: min=25.0, max=100000.0, zeros=0, negatives=0
term_in_months: min=1.0, max=158.0, zeros=0, negatives=0
lender_count: min=0, max=2986, zeros=3,383, negatives=0


In [ ]:
# Create a data dictionary containing column names, data types, missing values, and unique values
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "dtype": df.dtypes.values,
    "Data type": [
        type(df[col].iloc[0]).__name__
        for col in df.columns
    ],
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
    
})
data_dictionary

,Column,dtype,Data type,Missing Values,Unique Values
0,funded_amount,float64,float64,0,610
1,loan_amount,float64,float64,0,479
2,activity,str,str,0,163
3,sector,str,str,0,15
4,use,str,str,4233,423452
5,country_code,str,str,8,86
6,country,str,str,0,87
7,region,str,str,56800,12695
8,currency,str,str,0,67
9,term_in_months,float64,float64,0,148


### 3. Data Cleaning

The following cleaning steps are applied in sequence. 

#### Step 1 - Strip Whitespace from String Columns

String values in the raw file may carry leading or trailing whitespace. This is stripped uniformly across all object columns without losing any data.

In [ ]:
str_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip()
print(f"Stripped whitespace from {len(str_cols)} string columns: {str_cols}")

Stripped whitespace from 9 string columns: ['activity', 'sector', 'use', 'country_code', 'country', 'region', 'currency', 'borrower_genders', 'repayment_interval']


#### Step 2 - Duplicate Rows

Exact duplicate rows are dropped only if confirmed. A row is an exact duplicate if every column value matches another row. Kiva loan records are not expected to repeat; genuine duplicates are data entry errors.

In [ ]:
n_before = len(df)
df = df.drop_duplicates()
n_dropped = n_before - len(df)
print(f"Rows before: {n_before:,}")
print(f"Exact duplicates dropped: {n_dropped:,}")
print(f"Rows after: {len(df):,}")

Rows before: 671,205
Exact duplicates dropped: 25,579
Rows after: 645,626


#### Step 3 - Missing Values: `region`

`region` is a free-text field describing the sub-national area within a country. This step records the count; the fill to `'Not specified'` happens in Step 9 together with all remaining string columns.

In [ ]:
missing_region = df['region'].isnull().sum()
pct_region = missing_region / len(df) * 100
print(f"Missing region: {missing_region:,} ({pct_region:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

Missing region: 56,136 (8.7% of rows)
Will be filled with 'Not specified' in Step 9.


#### Step 4 - Missing Values: `borrower_genders`

`borrower_genders` holds one or more gender labels separated by commas. Missing values indicate the information was not supplied. This step records the count; the fill happens in Step 9.

In [ ]:
missing_bg = df['borrower_genders'].isnull().sum()
pct_bg = missing_bg / len(df) * 100
print(f"Missing borrower_genders: {missing_bg:,} ({pct_bg:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

Missing borrower_genders: 3,888 (0.6% of rows)
Will be filled with 'Not specified' in Step 9.


#### Step 5 - Missing Values: `use`

`use` is a free-text description of how the borrower intends to use the loan. This step records the count; the fill happens in Step 9.

In [ ]:
missing_use = df['use'].isnull().sum()
pct_use = missing_use / len(df) * 100
print(f"Missing use: {missing_use:,} ({pct_use:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

Missing use: 3,900 (0.6% of rows)
Will be filled with 'Not specified' in Step 9.


#### Step 6 - Missing Values: `country_code`

Eight rows have a missing `country_code` but a populated `country` value of 'Namibia'. The ISO 3166-1 alpha-2 code for Namibia is 'NA'. These eight rows are filled rather than dropped because the country is unambiguously known.

In [ ]:
missing_cc = df['country_code'].isnull().sum()
print(f"Missing country_code before fill: {missing_cc:,}")
print("Countries for these rows:")
print(df.loc[df['country_code'].isnull(), 'country'].value_counts().to_string())

df.loc[df['country_code'].isnull() & (df['country'] == 'Namibia'), 'country_code'] = 'NA'

remaining_cc = df['country_code'].isnull().sum()
print(f"\nMissing country_code after fill: {remaining_cc:,}")

Missing country_code before fill: 8
Countries for these rows:
country
Namibia    8

Missing country_code after fill: 0


#### Step 7 - Missing Values: `term_in_months`

`term_in_months` is checked for missing values. 

In [ ]:
missing_term = df['term_in_months'].isnull().sum()
pct_term = missing_term / len(df) * 100
print(f"Missing term_in_months: {missing_term:,} ({pct_term:.1f}% of rows)")

Missing term_in_months: 0 (0.0% of rows)


#### Step 8 - Remaining Missing Values Check

Columns with remaining NaN values are identified before the fill step.

In [ ]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if remaining_missing.empty:
    print("No NaN remaining before fill step.")
else:
    print("Columns with NaN before fill:")
    for col, cnt in remaining_missing.items():
        print(f"  {col}: {cnt:,} ({cnt/len(df)*100:.2f}%)")

Columns with NaN before fill:
  use: 3,900 (0.60%)
  region: 56,136 (8.69%)
  borrower_genders: 3,888 (0.60%)


#### Step 9 - Fill Remaining String NaN with 'Not specified'

All remaining NaN values in string columns are filled with `'Not specified'`. This makes the absence of data explicit and visible in groupby operations and visualizations instead of being silently excluded. The label `'Not specified'` is chosen over `'Unknown'` to avoid implying the information exists but is unclear - it simply was not recorded.

In [ ]:
str_cols_with_nan = df.select_dtypes(include=['object', 'str']).columns[
    df.select_dtypes(include=['object', 'str']).isnull().any()
].tolist()

for col in str_cols_with_nan:
    filled = df[col].isnull().sum()
    df[col] = df[col].fillna('Not specified')
    print(f"  {col}: filled {filled:,} NaN with 'Not specified'")

print(f"\nTotal NaN remaining in string columns: {df.select_dtypes(include=['object', 'str']).isnull().sum().sum()}")

  use: filled 3,900 NaN with 'Not specified'
  region: filled 56,136 NaN with 'Not specified'
  borrower_genders: filled 3,888 NaN with 'Not specified'

Total NaN remaining in string columns: 0


#### Step 10 - Numerical Validity

Records with a `loan_amount` of zero or below are not meaningful loan records. `funded_amount` can legitimately be zero (partially funded) but not negative. Rows failing these checks are dropped with a count reported.

In [ ]:
n_before = len(df)

invalid_loan = df['loan_amount'] <= 0
invalid_funded = df['funded_amount'] < 0
invalid_mask = invalid_loan | invalid_funded

print(f"Rows with loan_amount <= 0:  {invalid_loan.sum():,}")
print(f"Rows with funded_amount < 0: {invalid_funded.sum():,}")
print(f"Total invalid rows to drop:  {invalid_mask.sum():,}")

df = df[~invalid_mask].copy()
print(f"\nRows after numerical validity filter: {len(df):,} (dropped {n_before - len(df):,})")

Rows with loan_amount <= 0:  0
Rows with funded_amount < 0: 0
Total invalid rows to drop:  0

Rows after numerical validity filter: 645,626 (dropped 0)


#### Step 11 - Unusual `term_in_months` Values

The distribution of `term_in_months` is inspected to identify unusually short or long durations. Unusual values are noted but not removed unless they are clearly data entry errors (negative or zero), since genuine long-term loans exist.

In [ ]:
print("term_in_months distribution:")
print(df['term_in_months'].describe())
print(f"\nRows with term_in_months <= 0: {(df['term_in_months'] <= 0).sum():,}")
print(f"Rows with term_in_months > 120 (more than 10 years): {(df['term_in_months'] > 120).sum():,}")

term_in_months distribution:
count    645626.000000
mean         13.721718
std           8.477883
min           1.000000
25%           8.000000
50%          13.000000
75%          14.000000
max         158.000000
Name: term_in_months, dtype: float64

Rows with term_in_months <= 0: 0
Rows with term_in_months > 120 (more than 10 years): 208


Rows where `term_in_months` is zero or negative are dropped, as these cannot represent valid loan durations.

In [ ]:
n_before = len(df)
invalid_term = df['term_in_months'].notna() & (df['term_in_months'] <= 0)
df = df[~invalid_term].copy()
print(f"Dropped {n_before - len(df):,} rows with term_in_months <= 0.")
print(f"Rows remaining: {len(df):,}")

Dropped 0 rows with term_in_months <= 0.
Rows remaining: 645,626


#### Step 12 - Derived Feature: `funding_gap`

`funding_gap` is the difference between the requested amount and the funded amount. A positive value means the loan was partially funded. A negative value means more was disbursed than originally requested; the Kiva documentation confirms this can happen and these records are kept. Zero means fully funded.

In [ ]:
df['funding_gap'] = df['loan_amount'] - df['funded_amount']

fully_funded = (df['funding_gap'] == 0).sum()
partially_funded = (df['funding_gap'] > 0).sum()
over_funded = (df['funding_gap'] < 0).sum()

print(f"Fully funded (gap = 0):          {fully_funded:,} ({fully_funded/len(df)*100:.1f}%)")
print(f"Partially funded (gap > 0):      {partially_funded:,} ({partially_funded/len(df)*100:.1f}%)")
print(f"Over-funded (gap < 0):           {over_funded:,} ({over_funded/len(df)*100:.1f}%)")

print("\nfunding_gap statistics:")
print(df['funding_gap'].describe())

Fully funded (gap = 0):          597,599 (92.6%)
Partially funded (gap > 0):      48,025 (7.4%)
Over-funded (gap < 0):           2 (0.0%)

funding_gap statistics:
count    645626.000000
mean         58.499557
std         399.368658
min        -400.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       50000.000000
Name: funding_gap, dtype: float64


#### Step 13 - Final Cleaned Dataset Summary

In [ ]:
print("Final cleaned dataset:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print("\nRemaining missing values:")
final_missing = df.isnull().sum()
print(final_missing[final_missing > 0])
print("\nFirst rows:")
df.head(5)

Final cleaned dataset:
  Shape: (645626, 14)
  Columns: ['funded_amount', 'loan_amount', 'activity', 'sector', 'use', 'country_code', 'country', 'region', 'currency', 'term_in_months', 'lender_count', 'borrower_genders', 'repayment_interval', 'funding_gap']

Remaining missing values:
Series([], dtype: int64)

First rows:


,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,term_in_months,lender_count,borrower_genders,repayment_interval,funding_gap
0,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,12.0,12,female,irregular,0.0
1,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,11.0,14,"female, female",irregular,0.0
2,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,43.0,6,female,bullet,0.0
3,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,11.0,8,female,irregular,0.0
4,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,14.0,16,female,monthly,0.0


In [ ]:
# Display the shape of the DataFrame after cleaning
df.shape

(645626, 14)

In [ ]:
# Create a data dictionary containing column names, data types, missing values, and unique values after cleaning
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "dtype": df.dtypes.values,
    "Data type": [
        type(df[col].iloc[0]).__name__
        for col in df.columns
    ],
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
    
})
data_dictionary

,Column,dtype,Data type,Missing Values,Unique Values
0,funded_amount,float64,float64,0,610
1,loan_amount,float64,float64,0,479
2,activity,str,str,0,163
3,sector,str,str,0,15
4,use,str,str,0,417852
5,country_code,str,str,0,87
6,country,str,str,0,87
7,region,str,str,0,12696
8,currency,str,str,0,67
9,term_in_months,float64,float64,0,148


### 4. Save Cleaned Dataset

The cleaned and processed dataset is saved as a CSV to the `data` directory for use in subsequent analysis and visualization steps.

In [ ]:
df.to_csv(DATA_CLEANED_PATH)
print(f"Saved cleaned dataset to: {DATA_CLEANED_PATH}")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

KeyboardInterrupt: 

In [ ]:
# Make dfc available for the visualization and dashboard sections
dfc = df.copy()
print(f'dfc ready: {dfc.shape[0]:,} rows, {dfc.shape[1]} columns')


---

## Section 3: Exploratory Data Analysis and Visualizations 


### Visualization 1: Loan Count by Country


In [ ]:
TOP_N_COUNTRIES = 10
country_counts = dfc['country'].value_counts().head(TOP_N_COUNTRIES).reset_index()
country_counts.columns = ['country', 'count']
country_counts['pct'] = (country_counts['count'] / len(dfc) * 100).round(1)

fig_a1 = px.bar(
    country_counts,
    x='count',
    y='country',
    orientation='h',
    text=country_counts['pct'].map('{:.1f}%'.format),
    custom_data=['pct'],
    title=f'Top {TOP_N_COUNTRIES} Countries by Number of Kiva Loans',
    labels={'country': 'Country', 'count': 'Number of Loans'}
)
fig_a1.update_traces(
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Number of loans: %{x:,}<br>'
        '% of total number of loans: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_a1.show()


**Objective:** Identify which countries have the most Kiva loan records.

**What to look for:** Geographic concentration and the gap between the top country and the rest. 

**Configurable parameter:** `TOP_N_COUNTRIES` controls how many countries appear. Increase to see more context; decrease to focus on the biggest lenders.


### Visualization 2: Sector Distribution for top countries 


In [ ]:
# Number of top countries to display -- increase or decrease as needed
TOP_N_COUNTRIES = 10

top_countries = dfc['country'].value_counts().head(TOP_N_COUNTRIES).index.tolist()
top_df = dfc[dfc['country'].isin(top_countries)].copy()

sector_by_country = (
    top_df.groupby(['country', 'sector'])
    .size()
    .reset_index(name='count')
)
country_totals = top_df.groupby('country').size().reset_index(name='total')
sector_by_country = sector_by_country.merge(country_totals, on='country')
sector_by_country['pct'] = sector_by_country['count'] / sector_by_country['total'] * 100
sector_by_country['country_label'] = sector_by_country.apply(
    lambda r: f"{r['country']} ({r['total']:,})", axis=1
)

country_order = (
    top_df.groupby('country').size()
    .sort_values(ascending=False)
    .index.tolist()
)
label_order = [
    sector_by_country.loc[sector_by_country['country'] == c, 'country_label'].iloc[0]
    for c in country_order
]

# Sectors sorted by descending overall share across the full dataset
sector_order = (
    dfc['sector'].value_counts(normalize=True)
    .sort_values(ascending=False)
    .index.tolist()
)

fig_a2 = px.bar(
    sector_by_country,
    x='pct',
    y='country_label',
    color='sector',
    orientation='h',
    custom_data=['pct', 'country'],
    title=f'Sector Distribution for Top {TOP_N_COUNTRIES} Countries by Loan Count',
    labels={
        'pct': '% of Loans',
        'country_label': 'Country',
        'sector': 'Sector'
    },
    category_orders={'country_label': label_order, 'sector': sector_order}
)
fig_a2.update_traces(
    hovertemplate=(
        '<b>%{fullData.name}</b><br>'
        '% of total loans for %{customdata[1]}: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a2.update_layout(xaxis_ticksuffix='%')
fig_a2.show()



**Objective:** Show which sectors account for the largest share of loans in the countries with the highest lending activity.

**What to look for:** Countries where a single sector dominates vs. countries with a diversified sector mix. Differences in sector priorities between top-lending countries.

**Configurable parameter:** `TOP_N_COUNTRIES` controls how many countries appear. Increase to see more context; decrease to focus on the biggest lenders.


### Visualization 3: Underfunding by Sector 



In [ ]:
# Sector-level aggregation
sector_agg3 = (
    dfc.groupby('sector')
    .agg(
        loan_count=('loan_amount', 'count'),
        fully_funded=('funding_gap', lambda x: (x == 0).sum()),
        median_loan_amount=('loan_amount', 'median'),
        mean_loan_amount=('loan_amount', 'mean'),
        mean_gap=('funding_gap', 'mean')
    )
    .reset_index()
)
sector_agg3['pct_fully_funded'] = (sector_agg3['fully_funded'] / sector_agg3['loan_count'] * 100).round(2)

# Dynamic 5-bin coloring over the actual [min, max] range of pct_fully_funded
pct_lo = sector_agg3['pct_fully_funded'].min()
pct_hi = sector_agg3['pct_fully_funded'].max()
bin_step = (pct_hi - pct_lo) / 5
bin_edges  = [pct_lo + i * bin_step for i in range(6)]
bin_labels = [f'{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}%' for i in range(5)]
bin_colors = ['#dc2626', '#f87171', '#facc15', '#86efac', '#16a34a']
bin_color_map = dict(zip(bin_labels, bin_colors))

sector_agg3['funding_level'] = pd.cut(
    sector_agg3['pct_fully_funded'],
    bins=bin_edges,
    labels=bin_labels,
    include_lowest=True
).astype(str)

print('Funding level distribution:')
print(sector_agg3[['sector', 'pct_fully_funded', 'funding_level']].sort_values('pct_fully_funded').to_string())

fig_a3 = px.scatter(
    sector_agg3,
    x='loan_count',
    y='pct_fully_funded',
    size='median_loan_amount',
    color='funding_level',
    color_discrete_map=bin_color_map,
    text='sector',
    custom_data=['pct_fully_funded', 'mean_gap', 'loan_count', 'median_loan_amount', 'mean_loan_amount'],
    title='Funding Completeness by Sector',
    labels={
        'loan_count': 'Number of Loans',
        'pct_fully_funded': '% Fully Funded',
        'median_loan_amount': 'Median Loan Amount (USD)',
        'funding_level': 'Funding Level'
    },
    category_orders={'funding_level': bin_labels},
    size_max=60
)
fig_a3.update_traces(
    textposition='middle center',
    textfont_size=10,
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Number of loans: %{customdata[2]:,}<br>'
        '% fully funded: %{customdata[0]:.2f}%<br>'
        'Median loan size: $%{customdata[3]:,.0f}<br>'
        'Mean loan size: $%{customdata[4]:,.0f}<br>'
        'Average funding gap: $%{customdata[1]:,.2f}'
        '<extra></extra>'
    )
)
fig_a3.update_layout(template='plotly_white', height=600)
fig_a3.show()


Funding level distribution:
            sector  pct_fully_funded funding_level
8          Housing             86.74    86.7-89.2%
5    Entertainment             87.45    86.7-89.2%
13  Transportation             88.81    86.7-89.2%
2         Clothing             90.84    89.2-91.7%
12        Services             91.35    89.2-91.7%
11          Retail             91.66    89.2-91.7%
0      Agriculture             92.19    91.7-94.3%
7           Health             93.32    91.7-94.3%
6             Food             94.04    91.7-94.3%
10    Personal Use             94.29    94.3-96.8%
3     Construction             94.70    94.3-96.8%
14       Wholesale             96.69    94.3-96.8%
4        Education             97.44    96.8-99.3%
1             Arts             98.32    96.8-99.3%
9    Manufacturing             99.26    96.8-99.3%




**Objective:** 

Show whether loans in each sector are consistently fully funded, and how that relates to sector activity and typical loan size.

- **X-axis:** Number of loans (sector activity).
- **Y-axis:** Percentage of loans fully funded (funding success rate).
- **Bubble size:** Median loan amount (do larger loans struggle more to get fully funded?).
- **Bubble color:** Funding success rate binned into 5 levels from red to dark green.

**Hypothesis to test:** 
- Sectors with higher activity achieve more consistent full funding. 
    - Data shows no conclusive evidence. For example, manufacturing has lower activity but highest amount of fully funded loans. 

- Sectors with larger individual loans may show more partial funding. 
    - Data also shows no conclusive evidence here. For example, wholesale and construction has relatively high median loan sizes but on the higher end of fully funded loans. On the other hand retail has relatively smaller median loan sizes but does not achieve consistently full funding.


---

## Section 4: Dashboard: Kiva Lending Anomaly Analysis

An interactive dashboard is provided for deeper exploration of lending anomalies. 

**What is displayed:**
- For each selected sector, the top X countries are plotted by loan activity vs median loan size. 
- *Bubble area* encodes total funded amount; 
- *Bubble color* encodes the dominant repayment interval in that country for the selected sector. 

**What are possible anomalies:**

Countries with unusually high median loan amounts relative to their loan activity. There is unfortunately no records of loan defaults or partial repayments. However, loan repayment intervals can provide some insight - especially irregular loan repayment with high median loan amounts may indicate potential anomalies or even fraudulent activity.

After the cells in this section are run, **http://localhost:8051** can be opened in the browser.


In [394]:
import dash
from dash import dcc, html, Input, Output

# Dashboard 2 color scheme
BLUE_DEEP2   = '#1d4ed8'
BLUE_MID2    = '#3b82f6'
SIDEBAR_BG2  = '#dbeafe'
LABEL_COLOR2 = '#1e40af'

# Fixed color map for repayment intervals -- consistent across all sector selections
INTERVAL_COLORS = {
    'bullet':        "#f0c714",
    'irregular':     "#f91640",
    'monthly':       "#50e185",
    'weekly':        '#3b82f6',
    'Not specified': '#94a3b8'
}

dash2_df = dfc.copy()
sectors2 = sorted(dash2_df['sector'].unique())
dropdown_options2 = [{'label': 'Overall (all sectors)', 'value': 'Overall'}] + \
                    [{'label': s, 'value': s} for s in sectors2]

app2 = dash.Dash(__name__ + '_2',
    external_stylesheets=[
        'https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap'
    ]
)

app2.layout = html.Div([

    # Sidebar
    html.Div([
        html.H2('Kiva Lending Anomaly Analysis', style={
            'color': BLUE_DEEP2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '22px',
            'marginTop': '0',
            'marginBottom': '24px'
        }),
        html.Label('Select Sector:', style={
            'color': LABEL_COLOR2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '13px',
            'fontWeight': 'bold',
            'display': 'block',
            'marginBottom': '6px'
        }),
        dcc.Dropdown(
            id='d2-sector',
            options=dropdown_options2,
            value='Overall',
            clearable=False,
            style={'fontFamily': "'Inter', sans-serif", 'fontSize': '14px'}
        ),
        html.Div(style={'height': '16px'}),
        html.Label('Top N countries:', style={
            'color': LABEL_COLOR2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '13px',
            'fontWeight': 'bold',
            'display': 'block',
            'marginBottom': '6px'
        }),
        dcc.Input(
            id='d2-top-n',
            type='number',
            value=15,
            min=3,
            max=50,
            step=1,
            style={
                'width': '100%',
                'fontFamily': "'Inter', sans-serif",
                'fontSize': '14px',
                'padding': '6px 8px',
                'border': '1px solid #93c5fd',
                'borderRadius': '4px'
            }
        ),
        html.Hr(style={'borderColor': BLUE_MID2, 'margin': '24px 0'}),
        html.H4('Explanation', style={
            'color': BLUE_DEEP2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '16px',
            'marginBottom': '10px'
        }),
        html.P(
            'Countries with unusually high median loan amounts relative to their loan activity indicate potential anomalies.' 
            'Specifically, irregular loan repayment with high median loan amounts may indicate potential anomalies or even fraudulent activity.',
            style={
                'color': '#64748b',
                'fontFamily': "'Inter', sans-serif",
                'fontSize': '13px',
                'lineHeight': '1.6',
                'fontStyle': 'italic'
            }
        ),
        dcc.Markdown(
            '- For each selected sector, the top X countries are plotted by loan activity vs median loan size.\n'
            '- *Bubble area* encodes total funded amount;\n'
            '- *Bubble color* encodes the dominant repayment interval in that country for the selected sector.',
            style={
                'color': '#64748b',
                'fontFamily': "'Inter', sans-serif",
                'fontSize': '13px',
                'lineHeight': '1.8',
                'marginTop': '10px'
            }
        )
    ], style={
        'flex': '0 0 270px',
        'backgroundColor': SIDEBAR_BG2,
        'padding': '28px 20px',
        'borderRight': f'2px solid {BLUE_MID2}'
    }),

    # Chart
    html.Div(
        dcc.Graph(id='d2-scatter'),
        style={
            'flex': '1',
            'backgroundColor': '#f8fafc',
            'padding': '8px'
        }
    )

], style={
    'display': 'flex',
    'fontFamily': "'Inter', sans-serif",
    'maxWidth': '1400px',
    'margin': '20px auto',
    'boxShadow': '0 4px 14px rgba(0,0,0,0.12)',
    'borderRadius': '6px',
    'overflow': 'hidden',
    'minHeight': '680px'
})


In [395]:


@app2.callback(
    Output('d2-scatter', 'figure'),
    Input('d2-sector', 'value'),
    Input('d2-top-n', 'value')
)
def update_sector_scatter(selected_sector, top_n_raw):
    top_n = max(3, min(50, int(top_n_raw or 15)))

    if selected_sector == 'Overall':
        sector_data = dash2_df
        chart_title = f'Loan Activity vs Median Loan Size: All Sectors (Top {top_n} Countries)'
    else:
        sector_data = dash2_df[dash2_df['sector'] == selected_sector]
        chart_title = f'Loan Activity vs Median Loan Size: {selected_sector} (Top {top_n} Countries)'

    country_agg = (
        sector_data.groupby('country')
        .agg(
            loan_count=('loan_amount', 'count'),
            median_loan_amount=('loan_amount', 'median'),
            mean_loan_amount=('loan_amount', 'mean'),
            total_funded_amount=('funded_amount', 'sum')
        )
        .reset_index()
    )

    dominant_interval = (
        sector_data.groupby(['country', 'repayment_interval'])
        .size()
        .reset_index(name='cnt')
        .sort_values('cnt', ascending=False)
        .drop_duplicates('country')
        .set_index('country')['repayment_interval']
    )
    country_agg['dominant_interval'] = country_agg['country'].map(dominant_interval)

    country_agg = (
        country_agg.nlargest(top_n, 'loan_count')
        .reset_index(drop=True)
    )

    fig = px.scatter(
        country_agg,
        x='loan_count',
        y='median_loan_amount',
        size='total_funded_amount',
        color='dominant_interval',
        color_discrete_map=INTERVAL_COLORS,
        text='country',
        custom_data=['total_funded_amount', 'dominant_interval', 'mean_loan_amount'],
        title=chart_title,
        labels={
            'loan_count': 'Number of Loans',
            'median_loan_amount': 'Median Loan Amount (USD)',
            'total_funded_amount': 'Total Funded (USD)',
            'dominant_interval': 'Dominant Repayment Interval'
        },
        size_max=60
    )
    fig.update_traces(
        textposition='middle center',
        textfont_size=10,
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Dominant interval: %{customdata[1]}<br>'
            'Loan count: %{x:,}<br>'
            'Median loan amount: $%{y:,.0f}<br>'
            'Mean loan amount: $%{customdata[2]:,.0f}<br>'
            'Total funded: $%{customdata[0]:,.0f}'
            '<extra></extra>'
        )
    )
    fig.update_layout(
        template='plotly_white',
        plot_bgcolor='#eff6ff',
        title_font_size=16,
        title_font_family='Inter, sans-serif',
        height=620
    )
    return fig


In [396]:


app2.run(debug=False, port=8051, jupyter_mode='external')


Dash app running on http://127.0.0.1:8051/
